[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gouravkhanijoe13/agentic-ai-lab/blob/main/Lesson_75_Feedback_Loops_Data_Flywheel.ipynb)

# Lesson 75 — Feedback Loops & the Data Flywheel

**Phase 8: Production Ops for LLM Systems — Lesson 5 of ~6**

| # | Lesson | Status |
|---|--------|--------|
| 71 | Observability & online evals | done |
| 72 | Structured logging & trace search | done |
| 73 | Alerting, SLOs & on-call | done |
| 74 | A/B testing & guarded rollouts | done |
| **75** | **Feedback loops & the data flywheel** | **today** |
| 76 | Phase 8 capstone — ship `agent-bench-obs` | next |

---

## The one idea

Lessons 71–74 built machinery that **watches** production:

- **L71** metrics say *something is wrong*
- **L72** trace search says *what* is wrong
- **L73** alerts say *a human should look now*
- **L74** rollouts say *this new version is worse — pull it*

Every one of those produced a by-product you have been **throwing away**: a
stream of production requests annotated with evidence about whether they went
well or badly.

> **A feedback loop is the pipe that turns that exhaust into labeled data.
> A data flywheel is what you get when the labeled data makes the system
> better, which produces better data, which makes it better still.**

Without the loop, your `agent-bench` golden set is frozen at whatever you
imagined on day one. With the loop, it grows toward **the failures your real
users actually hit** — and that is the difference between an eval suite that
scores 98% forever and one that keeps finding bugs.

### The loop, drawn

```
        production traffic
               |
   [L71/L72/L73/L74 signals]        <- thumbs, retries, edits, alerts, losing arms
               |
        weak labeling               <- fuse noisy signals into ONE guess
               |
      active-learning sampling      <- spend the scarce human-review budget well
               |
       human review (small!)        <- the only real ground truth in the picture
               |
     dataset hygiene + dedup        <- redact PII, refuse contamination
               |
      GOLDEN SET grows  ----------> agent-bench catches the new failure
               |
      prompt / model improvement
               |
      guarded rollout (L74)  -------> back to production traffic
```

Today we build every box in that diagram, with real code and hard assertions.

## Setup

Deterministic, offline, no API key. Everything is simulated with a fixed seed so
your numbers match the ones in the text exactly.

In [ ]:
!pip install rich -q

import hashlib, json, math, os, random, re, sys, textwrap
from dataclasses import dataclass, field, asdict
from datetime import datetime, timedelta, timezone
from typing import Any, Dict, List, Optional, Sequence, Tuple

from rich.console import Console
from rich.table import Table

console = Console(force_jupyter=False, no_color=False, highlight=False)

SEED = 75
NOW = datetime(2026, 7, 21, 12, 0, 0, tzinfo=timezone.utc)
BASE = "/content"
os.makedirs(BASE, exist_ok=True)

random.seed(SEED)
print("Lesson 75 setup OK  |  seed =", SEED, " NOW =", NOW.isoformat(), " BASE =", BASE)

## 1. The signal taxonomy — what production actually gives you

There are three families of signal, and confusing them is the single most
common way teams poison their own dataset.

| Family | Examples | Coverage | Bias | Trustworthiness |
|---|---|---|---|---|
| **Explicit** | thumbs up/down, star rating, "report a problem" | **tiny** (1–10% of traffic) | **severe** — angry users rate, satisfied users leave | high per-datapoint |
| **Implicit** | regenerate/retry, user edits the answer, abandons the session, copies the output | **near 100%** | moderate — a retry can mean "wrong" or "give me another angle" | low per-datapoint |
| **System** | L73 guardrail fired, L74 losing arm, exception, timeout, schema-validation failure | **100%** | low | high, but only covers *hard* failures |

The two rules that follow from this table:

1. **Never build a dataset from explicit feedback alone.** Thumbs-down data is
   a survey of your angriest 3%. Optimizing for it optimizes for them.
2. **Never treat any single signal as a label.** A retry is *evidence*, not
   truth. You fuse many weak signals — that is the next section.

There is a fourth family worth naming because it is free and people forget it:
**L74's losing arm is labeled data.** When a challenger gets rolled back, every
request it served is a sample of a known-worse policy. That is a gift.

In [ ]:
TOPICS = ["summarize", "extract", "classify", "translate", "multi_table_math"]
TOPIC_W = [0.30, 0.25, 0.20, 0.15, 0.10]

# HIDDEN ground truth of how often each topic actually goes wrong.
# In real life you do NOT have this column - it exists here only so we can
# measure how good our weak labels are.
BAD_RATE = {"summarize": 0.06, "extract": 0.06, "classify": 0.05,
            "translate": 0.07, "multi_table_math": 0.72}

PII_SUFFIX = " contact ops@example.com key sk-live-A1B2C3D4E5"

@dataclass
class FeedbackEvent:
    req_id: str
    user_id: str
    topic: str
    text: str
    heuristic_score: float
    judge_score: Optional[float]
    thumbs: Optional[int]
    regenerated: bool
    edited: bool
    abandoned: bool
    guardrail_fired: bool
    arm: str
    is_bad: bool          # <-- HIDDEN. teaching only. never available in prod.

def simulate_traffic(n: int = 400, seed: int = SEED, tag: str = "") -> List[FeedbackEvent]:
    rng = random.Random(seed)
    out: List[FeedbackEvent] = []
    for i in range(n):
        topic = rng.choices(TOPICS, weights=TOPIC_W, k=1)[0]
        bad = rng.random() < BAD_RATE[topic]

        heur = rng.uniform(0.15, 0.58) if bad else rng.uniform(0.62, 1.0)
        # L71 rule: the LLM judge runs on a ~30% sample, off the hot path.
        if rng.random() < 0.30:
            judge = rng.uniform(0.05, 0.52) if bad else rng.uniform(0.60, 1.0)
        else:
            judge = None

        # explicit feedback: rare, and biased toward the unhappy
        thumbs = None
        if bad and rng.random() < 0.18:
            thumbs = -1 if rng.random() < 0.90 else 1
        elif (not bad) and rng.random() < 0.04:
            thumbs = 1 if rng.random() < 0.85 else -1

        regenerated = rng.random() < (0.55 if bad else 0.05)
        edited      = rng.random() < (0.35 if bad else 0.08)
        abandoned   = rng.random() < (0.25 if bad else 0.03)
        guardrail   = rng.random() < (0.20 if bad else 0.005)

        text = topic + ": task variant " + str(i % 37) + (" " + tag if tag else "")
        if i % 25 == 0:
            text = text + PII_SUFFIX            # PII shows up in real logs

        out.append(FeedbackEvent(
            req_id="req_" + str(i).zfill(4),
            user_id="user_" + str(rng.randrange(60)).zfill(3),
            topic=topic, text=text,
            heuristic_score=heur, judge_score=judge, thumbs=thumbs,
            regenerated=regenerated, edited=edited, abandoned=abandoned,
            guardrail_fired=guardrail,
            arm="challenger" if rng.random() < 0.10 else "champion",
            is_bad=bad,
        ))
    return out

EVENTS = simulate_traffic(400)

n = len(EVENTS)
n_bad = sum(e.is_bad for e in EVENTS)
n_thumbs = sum(e.thumbs is not None for e in EVENTS)
n_implicit = sum(e.regenerated or e.edited or e.abandoned for e in EVENTS)
n_system = sum(e.guardrail_fired for e in EVENTS)
thumbs_down_bad = sum(1 for e in EVENTS if e.thumbs == -1 and e.is_bad)
thumbs_down = sum(1 for e in EVENTS if e.thumbs == -1)

t = Table(title="Signal coverage on 400 production requests", header_style="bold")
for c in ["family", "requests carrying it", "coverage"]:
    t.add_column(c)
t.add_row("explicit (any thumbs)", str(n_thumbs), "{:.1%}".format(n_thumbs / n))
t.add_row("implicit (retry/edit/abandon)", str(n_implicit), "{:.1%}".format(n_implicit / n))
t.add_row("system (guardrail fired)", str(n_system), "{:.1%}".format(n_system / n))
t.add_row("[dim]hidden truth: actually bad[/dim]", str(n_bad), "{:.1%}".format(n_bad / n))
console.print(t)

print()
print("Explicit feedback is RARE ....... {:.1%} of traffic".format(n_thumbs / n))
print("...and BIASED: of the {} thumbs-down, {} were genuinely bad ({:.0%} precision),".format(
    thumbs_down, thumbs_down_bad, thumbs_down_bad / max(thumbs_down, 1)))
print("   but they cover only {:.1%} of all {} bad responses (recall).".format(
    thumbs_down_bad / n_bad, n_bad))

COVERAGE_OK = (n_thumbs / n) < 0.15 and (n_implicit / n) > (n_thumbs / n)
EXPLICIT_RECALL = thumbs_down_bad / n_bad
assert COVERAGE_OK, "explicit feedback should be rarer than implicit"
assert EXPLICIT_RECALL < 0.40, "thumbs alone must miss most failures - that is the point"
print("\nOK - explicit feedback alone would miss {:.0%} of your failures.".format(1 - EXPLICIT_RECALL))

## 2. Weak labeling — fusing noisy signals into one guess

You have five noisy opinions about each request and zero ground truth. The
technique for combining them is **weak supervision**: assign each signal a
weight (how much it moves your belief that the response was bad), sum the
weights, squash through a logistic, and read off a probability.

```
score  = sum(weight[s] for s in signals_present)
p_bad  = 1 / (1 + exp(-(score - bias)))
```

The `bias` term encodes your prior. We use `-1.0`, which says: **silence means
probably fine**. Most requests generate no signal at all, and you must not
label all of them "bad" just because nobody clapped.

Three outputs, not two:

| p_bad | label | what you do with it |
|---|---|---|
| ≥ 0.75 | `bad` | strong candidate for the eval set (still needs human confirmation) |
| ≤ 0.30 | `good` | leave it alone |
| in between | `unknown` | **this is the interesting bucket** — see §3 |

The `unknown` bucket is not a failure of the method. It is the method telling
you where your scarce human attention is worth the most.

In [ ]:
WEIGHTS = {
    "thumbs_down": 3.0, "thumbs_up": -3.0,
    "regenerated": 1.5, "edited": 1.0, "abandoned": 1.2,
    "guardrail_fired": 2.5,
    "low_heuristic": 1.0, "low_judge": 2.0, "high_judge": -1.5,
}
BIAS = -1.0
LO, HI = 0.30, 0.75

def active_signals(e: FeedbackEvent) -> List[str]:
    s = []
    if e.thumbs == -1: s.append("thumbs_down")
    if e.thumbs == 1:  s.append("thumbs_up")
    if e.regenerated:  s.append("regenerated")
    if e.edited:       s.append("edited")
    if e.abandoned:    s.append("abandoned")
    if e.guardrail_fired: s.append("guardrail_fired")
    if e.heuristic_score < 0.55: s.append("low_heuristic")
    if e.judge_score is not None and e.judge_score < 0.50: s.append("low_judge")
    if e.judge_score is not None and e.judge_score >= 0.85: s.append("high_judge")
    return s

def logistic(x: float) -> float:
    return 1.0 / (1.0 + math.exp(-x))

@dataclass
class WeakLabel:
    label: str
    p_bad: float
    signals: List[str]

def weak_label(e: FeedbackEvent) -> WeakLabel:
    sigs = active_signals(e)
    score = sum(WEIGHTS.get(s, 0.0) for s in sigs) + BIAS
    p = logistic(score)
    lab = "bad" if p >= HI else ("good" if p <= LO else "unknown")
    return WeakLabel(lab, p, sigs)

LABELS = {e.req_id: weak_label(e) for e in EVENTS}

buckets = {"bad": 0, "good": 0, "unknown": 0}
for wl in LABELS.values():
    buckets[wl.label] += 1

pred_bad = [e for e in EVENTS if LABELS[e.req_id].label == "bad"]
tp = sum(e.is_bad for e in pred_bad)
WEAK_PRECISION = tp / max(len(pred_bad), 1)
WEAK_RECALL = tp / n_bad

t = Table(title="Weak label quality (measured against the hidden truth)", header_style="bold")
for c in ["bucket", "count", "share"]:
    t.add_column(c)
for k in ["bad", "unknown", "good"]:
    t.add_row(k, str(buckets[k]), "{:.1%}".format(buckets[k] / n))
console.print(t)

print()
print("precision of the 'bad' bucket .. {:.1%}  (when we cry wolf, is there a wolf?)".format(WEAK_PRECISION))
print("recall of the 'bad' bucket ..... {:.1%}  (what share of real failures did we catch?)".format(WEAK_RECALL))
print("vs thumbs-down alone: recall ... {:.1%}".format(EXPLICIT_RECALL))

# 3 example events, worst signals first
ex = sorted(EVENTS, key=lambda e: -LABELS[e.req_id].p_bad)[:3]
t2 = Table(title="Three highest-p_bad requests", header_style="bold")
for c in ["req_id", "topic", "p_bad", "signals", "truly bad?"]:
    t2.add_column(c)
for e in ex:
    wl = LABELS[e.req_id]
    t2.add_row(e.req_id, e.topic, "{:.2f}".format(wl.p_bad), ",".join(wl.signals), str(e.is_bad))
console.print(t2)

assert WEAK_PRECISION > 0.70, "weak labels are noisy but should not be garbage"
assert WEAK_RECALL > EXPLICIT_RECALL, "fusing signals must beat thumbs alone"
assert WEAK_RECALL < 1.0, "weak labels ALWAYS miss silent failures - never assume full recall"
assert buckets["unknown"] > 0, "there must be an uncertain bucket to spend review budget on"
print("\nOK - fusion beats any single signal, and still misses {:.0%} of failures.".format(1 - WEAK_RECALL))

## 3. Active learning — spending a tiny human budget well

Here is the real constraint. You have 400 requests and a reviewer who will
label **40**. Which 40?

| Strategy | What it picks | Yield (share truly bad) | Information gained |
|---|---|---|---|
| `random` | 40 at random | ≈ the base rate | low — you re-confirm what you knew |
| `confident_bad` | the 40 highest `p_bad` | **very high** | **low** — you already knew these were bad |
| `uncertainty` | the 40 the model is least sure about | middling | **highest** — every label resolves a real question |

This is the counter-intuitive part, and it is why "yield" is a trap metric.
Labelling 40 obvious failures teaches your system almost nothing: `p_bad` was
already 0.95. Labelling 40 examples sitting at `p_bad ≈ 0.5` — or where the
cheap heuristic and the expensive judge **disagree** — resolves 40 genuine
open questions and re-calibrates the whole weak labeler.

Our uncertainty score has two ingredients:

```
uncertainty = 0.6 * (1 - 2*|p_bad - 0.5|)      # label ambiguity
            + 0.4 * |heuristic_score - judge_score|   # scorer disagreement
```

In practice you run a **blend**: some `confident_bad` (to mine known failure
modes into the eval set), some `uncertainty` (to improve the labeler), and a
**mandatory random slice** (to keep an unbiased estimate of production quality —
§6 explains why skipping this is fatal).

In [ ]:
def uncertainty(e: FeedbackEvent) -> float:
    wl = LABELS[e.req_id]
    label_unc = 1.0 - 2.0 * abs(wl.p_bad - 0.5)
    disagree = 0.0 if e.judge_score is None else abs(e.heuristic_score - e.judge_score)
    return 0.6 * label_unc + 0.4 * disagree

def sample_for_review(events, budget, strategy, seed=SEED):
    if strategy == "random":
        pool = list(events); random.Random(seed).shuffle(pool); return pool[:budget]
    if strategy == "confident_bad":
        return sorted(events, key=lambda e: -LABELS[e.req_id].p_bad)[:budget]
    if strategy == "uncertainty":
        return sorted(events, key=lambda e: -uncertainty(e))[:budget]
    raise ValueError(strategy)

BUDGET = 40
res = {}
for strat in ["random", "confident_bad", "uncertainty"]:
    pick = sample_for_review(EVENTS, BUDGET, strat)
    yld = sum(e.is_bad for e in pick) / BUDGET
    info = sum(1.0 - 2.0 * abs(LABELS[e.req_id].p_bad - 0.5) for e in pick) / BUDGET
    topics = len({e.topic for e in pick})
    res[strat] = {"yield": yld, "info": info, "topics": topics}

t = Table(title="40 human labels - three ways to spend them", header_style="bold")
for c in ["strategy", "yield (truly bad)", "mean label ambiguity", "topics covered"]:
    t.add_column(c)
for k, v in res.items():
    t.add_row(k, "{:.1%}".format(v["yield"]), "{:.2f}".format(v["info"]), str(v["topics"]))
console.print(t)

assert res["confident_bad"]["yield"] > res["random"]["yield"], "targeting must beat random on yield"
assert res["uncertainty"]["info"] > res["random"]["info"], "uncertainty sampling must beat random on information"
assert res["uncertainty"]["info"] > res["confident_bad"]["info"], \
    "high-yield sampling is LOW information - that is the lesson"
print("\nOK - 'confident_bad' wins on yield, 'uncertainty' wins on information. They are different jobs.")

# The blend you would actually run in production: mine failures, resolve
# ambiguity, AND keep an unbiased random slice (see section 6 for why).
def blended_review(events, budget, seed=SEED):
    n_conf = int(budget * 0.4)
    n_unc = int(budget * 0.4)
    n_rand = budget - n_conf - n_unc
    chosen, seen = [], set()
    for strat, want in [("confident_bad", n_conf), ("uncertainty", n_unc), ("random", n_rand)]:
        taken = 0
        for e in sample_for_review(events, len(events), strat, seed):
            if taken >= want:
                break
            if e.req_id in seen:
                continue
            seen.add(e.req_id)
            chosen.append((strat, e))
            taken += 1
    return chosen

BLEND = blended_review(EVENTS, BUDGET)
mix = {}
for strat, e in BLEND:
    mix[strat] = mix.get(strat, 0) + 1
print("blended review batch:", mix, " total:", len(BLEND))
assert len(BLEND) == BUDGET and len(mix) == 3, "blend must draw from all three strategies"

## 4. Dataset hygiene — the three ways a mined dataset gets poisoned

The reviewed batch is now real ground truth. Before it touches your golden set,
three guards must run. Skipping any of them silently destroys the value of your
eval suite, and you will not notice for months.

1. **Redaction.** Production text carries emails, API keys, customer names. An
   eval set is a file that gets committed to git, pasted into issues, and shared
   with contributors. Redact on the *write* path (this is L72's `redact()`).

2. **Deduplication.** Production traffic is enormously repetitive. Ten copies of
   the same prompt in your eval set means that one case now carries 10× weight
   in your headline score. Hash on the *normalized* text.

3. **Contamination.** The killer. If a case is already in the golden set and you
   add it again as *training* or *few-shot* material — or if you tune the prompt
   on exactly the cases you then score against — your eval number becomes a
   measure of memorization. `agent-bench` scores stop meaning anything.
   **The golden set must be write-once-and-frozen with respect to anything you
   optimize against.**

In [ ]:
PII_PATTERNS = [
    re.compile(r"[\w\.\-\+]+@[\w\-]+\.[\w\.\-]+"),
    re.compile(r"sk-[A-Za-z0-9\-_]{6,}"),
]

def redact(text: str) -> str:
    for p in PII_PATTERNS:
        text = p.sub("[REDACTED]", text)
    return text

def normalize(text: str) -> str:
    return re.sub(r"\s+", " ", text.strip().lower())

def content_hash(text: str) -> str:
    return hashlib.md5(normalize(text).encode("utf-8")).hexdigest()[:16]

@dataclass
class EvalCase:
    case_id: str
    prompt: str
    topic: str
    label: str
    source: str
    provenance: Dict[str, Any]

class GoldenSet:
    def __init__(self, cases=None):
        self.cases = list(cases or [])
        self._hashes = {content_hash(c.prompt) for c in self.cases}
    def __len__(self): return len(self.cases)
    def contains(self, prompt): return content_hash(prompt) in self._hashes
    def add(self, case):
        h = content_hash(case.prompt)
        if h in self._hashes:
            return False
        self._hashes.add(h); self.cases.append(case); return True
    def topics(self):
        out = {}
        for c in self.cases:
            out[c.topic] = out.get(c.topic, 0) + 1
        return out

def build_cases(reviewed, golden, source="production"):
    accepted, stats, seen_batch = [], {"seen": 0, "already_in_golden": 0, "dup_in_batch": 0, "accepted": 0}, set()
    for ev, human in reviewed:
        stats["seen"] += 1
        prompt = redact(ev.text)
        h = content_hash(prompt)
        if golden.contains(prompt):
            stats["already_in_golden"] += 1; continue
        if h in seen_batch:
            stats["dup_in_batch"] += 1; continue
        seen_batch.add(h)
        accepted.append(EvalCase("case_" + h, prompt, ev.topic, human, source,
                                 {"req_id": ev.req_id, "signals": active_signals(ev), "arm": ev.arm}))
        stats["accepted"] += 1
    return accepted, stats

# A golden set that already exists from day one (hand-written seeds),
# plus one case that happens to duplicate a production request.
SEED_CASES = [EvalCase("seed_" + str(i), TOPICS[i % len(TOPICS)] + ": handwritten seed " + str(i),
                       TOPICS[i % len(TOPICS)], "good", "seed", {}) for i in range(8)]
GOLDEN = GoldenSet(SEED_CASES)
GOLDEN.add(EvalCase("seed_dup", redact(EVENTS[0].text), EVENTS[0].topic, "bad", "seed", {}))
print("golden set starts at", len(GOLDEN), "cases")

# The human reviews the blended batch. We SIMULATE the human by revealing
# the hidden truth - in real life this is a person in a labeling UI.
reviewed = [(e, "bad" if e.is_bad else "good") for _strat, e in BLEND]
reviewed.append((EVENTS[0], "bad"))     # already in golden -> contamination guard
reviewed.append((EVENTS[1], "good"))
reviewed.append((EVENTS[1], "good"))    # exact duplicate -> dedup guard

new_cases, stats = build_cases(reviewed, GOLDEN)
t = Table(title="Dataset hygiene report", header_style="bold")
for c in ["stage", "count"]:
    t.add_column(c)
for k, v in stats.items():
    t.add_row(k, str(v))
console.print(t)

pii_leaked = [c for c in new_cases if "@" in c.prompt or "sk-" in c.prompt]
added = sum(GOLDEN.add(c) for c in new_cases)
print("\nPII leaked into the dataset:", len(pii_leaked))
print("cases added to golden set  :", added, "-> golden set now", len(GOLDEN))
print("golden topic mix           :", GOLDEN.topics())

assert stats["already_in_golden"] >= 1, "contamination guard must have fired"
assert stats["dup_in_batch"] >= 1, "dedup guard must have fired"
assert len(pii_leaked) == 0, "no PII may reach the eval set"
assert len({c.case_id for c in GOLDEN.cases}) == len(GOLDEN.cases), "golden set must have unique cases"
print("\nOK - redaction, dedup and contamination guards all fired.")

## 5. The payoff — does a mined eval set actually catch more?

Everything so far is plumbing. Here is the claim it exists to support:

> **An eval set mined from production feedback detects real improvements that a
> randomly-sampled eval set of the same size cannot see.**

The experiment. Two versions of the agent:

- **v1** — the current one. It falls apart on `multi_table_math` (80% failure)
  and is fine everywhere else (5% failure).
- **v2** — a prompt fix specifically targeting `multi_table_math` (down to 15%).

Two eval sets, **both of size 60**:

- **`random_set`** — 60 cases sampled uniformly from production. Its topic mix
  mirrors traffic, so only ~10% of it is `multi_table_math`.
- **`mined_set`** — 60 cases from today's feedback pipeline. Because the
  pipeline routes on `p_bad` and uncertainty, it is *concentrated* on the
  failure mode.

If the mined set is doing its job, `v2 - v1` should be a big, unmissable
improvement on `mined_set` and statistical mush on `random_set`. The same fix,
the same agent — only the measuring instrument differs.

In [ ]:
FAIL_RATE = {
    "v1": {"multi_table_math": 0.80, "_default": 0.05},
    "v2": {"multi_table_math": 0.15, "_default": 0.05},
}

def run_eval(cases, version, seed=SEED):
    rng = random.Random(seed)
    fr = FAIL_RATE[version]
    passed = 0
    for c in cases:
        p_fail = fr.get(c.topic, fr["_default"])
        if rng.random() >= p_fail:
            passed += 1
    return passed / max(len(cases), 1)

SET_SIZE = 60
random_pool = list(EVENTS)
random.Random(SEED + 1).shuffle(random_pool)
random_set = [EvalCase("r" + str(i), redact(e.text), e.topic, "?", "random", {})
              for i, e in enumerate(random_pool[:SET_SIZE])]

mined_pool = sorted(EVENTS, key=lambda e: -(0.7 * LABELS[e.req_id].p_bad + 0.3 * uncertainty(e)))
mined_set = [EvalCase("m" + str(i), redact(e.text), e.topic, "?", "mined", {})
             for i, e in enumerate(mined_pool[:SET_SIZE])]

def topic_share(cases, topic):
    return sum(1 for c in cases if c.topic == topic) / len(cases)

rows = []
for name, cs in [("random_set", random_set), ("mined_set", mined_set)]:
    p1, p2 = run_eval(cs, "v1"), run_eval(cs, "v2")
    rows.append((name, topic_share(cs, "multi_table_math"), p1, p2, p2 - p1))

t = Table(title="Same fix, two measuring instruments (n=60 each)", header_style="bold")
for c in ["eval set", "% multi_table_math", "v1 pass", "v2 pass", "detected lift"]:
    t.add_column(c)
for name, share, p1, p2, d in rows:
    t.add_row(name, "{:.0%}".format(share), "{:.1%}".format(p1), "{:.1%}".format(p2),
              "[bold]{:+.1%}[/bold]".format(d))
console.print(t)

RANDOM_LIFT = rows[0][4]
MINED_LIFT = rows[1][4]
print()
print("random_set says the fix is worth {:+.1%} - indistinguishable from noise on 60 samples.".format(RANDOM_LIFT))
print("mined_set  says the fix is worth {:+.1%} - unmissable.".format(MINED_LIFT))
print("Same agent. Same fix. The eval set was the bottleneck.")

assert MINED_LIFT > RANDOM_LIFT, "the mined set must be more sensitive to the real fix"
assert MINED_LIFT > 0.25, "mined set should show a large, obvious lift"
assert RANDOM_LIFT < 0.20, "random set of this size should barely see it"
print("\nOK - THE FLYWHEEL PAYOFF: production feedback bought a sharper instrument.")

## 6. How feedback loops go wrong

A flywheel that spins the wrong way is worse than no flywheel, because it
accelerates. Four documented failure modes:

**Survivorship bias.** You only ever collect data on requests the system
*accepted*. Everything it refused, truncated, or timed out is invisible, so the
next version is optimized on a distribution that excludes its own worst
behavior.

**Distribution skew.** Mined data is, by construction, *not* production data —
it is production's tail. Optimize purely on it and you get a model that is
brilliant at `multi_table_math` and quietly worse at the 90% of traffic you
stopped measuring. **Fix: always keep a random, unbiased holdout slice** — the
`n_rand` in §3's blend is not a rounding error, it is the control group.

**Model collapse / self-training rot.** If your labels come from an LLM judge
and your improvements come from those labels, the system is grading its own
homework. Each generation drifts further from human preference while the
metrics look great. **Fix: periodically re-calibrate the judge against fresh
human labels, and never let a generation of purely synthetic labels feed
directly into the next.**

**Feedback-shaped Goodhart.** You optimize for "fewer retries." The model learns
to produce long, confident-sounding answers that discourage retries without
being more correct. **The signal was a proxy; you optimized the proxy.**

The measurement below quantifies skew: total-variation distance between the
mined set's topic distribution and production's.

In [ ]:
def topic_dist(topics_list):
    d = {}
    for tt in topics_list:
        d[tt] = d.get(tt, 0) + 1
    total = sum(d.values())
    return {k: v / total for k, v in d.items()}

def tv_distance(p, q):
    keys = set(p) | set(q)
    return 0.5 * sum(abs(p.get(k, 0.0) - q.get(k, 0.0)) for k in keys)

prod_dist = topic_dist([e.topic for e in EVENTS])
mined_dist = topic_dist([c.topic for c in mined_set])
rand_dist = topic_dist([c.topic for c in random_set])

# The fix: mined cases for sensitivity + a random holdout for an honest estimate.
combined = mined_set[:40] + random_set[:40]
comb_dist = topic_dist([c.topic for c in combined])

TV_MINED = tv_distance(mined_dist, prod_dist)
TV_RANDOM = tv_distance(rand_dist, prod_dist)
TV_COMBINED = tv_distance(comb_dist, prod_dist)

t = Table(title="Distribution skew vs production traffic", header_style="bold")
for c in ["set", "multi_table_math share", "TV distance from prod", "reads as"]:
    t.add_column(c)
t.add_row("production", "{:.0%}".format(prod_dist.get("multi_table_math", 0)), "0.00", "the truth")
t.add_row("random_set", "{:.0%}".format(rand_dist.get("multi_table_math", 0)),
          "{:.2f}".format(TV_RANDOM), "unbiased, insensitive")
t.add_row("mined_set", "{:.0%}".format(mined_dist.get("multi_table_math", 0)),
          "{:.2f}".format(TV_MINED), "[bold]sensitive, badly skewed[/bold]")
t.add_row("mined + random holdout", "{:.0%}".format(comb_dist.get("multi_table_math", 0)),
          "{:.2f}".format(TV_COMBINED), "sensitive AND anchored")
console.print(t)

print()
print("If you report the mined set's pass rate as 'production quality', you are")
print("reporting the quality of your hardest {:.0%} of traffic. That number is not wrong -".format(
    mined_dist.get("multi_table_math", 0)))
print("it is answering a different question than the one your VP asked.")

assert TV_MINED > TV_RANDOM, "mined data is skewed by construction - know it"
assert TV_MINED > 4 * TV_RANDOM, "the mined set should be dramatically more skewed than a random one"
assert TV_COMBINED < TV_MINED, "adding a random holdout must pull you back toward reality"
print("\nOK - skew measured, and the random-holdout fix demonstrably reduces it.")

## 7. Turning the wheel — three full revolutions

One pass is a feedback loop. Repeating it is a flywheel. Each turn:

1. serve traffic, collect signals
2. weak-label, sample under a fixed human budget
3. hygiene, add to golden set
4. re-run `agent-bench`, fix the top failure mode
5. **guarded rollout (L74)** — because a fix is a change, and changes get ramped

Be precise about what compounds, because it is easy to overclaim here. The
detected *lift* on the `v1 → v2` fix does **not** keep rising — once your golden
set contains enough `multi_table_math` cases to see that bug, more of them add
nothing. Saturating on a bug you have already caught is the correct behaviour.

What actually compounds is **statistical power**:

```
standard error of your headline pass rate  =  sqrt( p*(1-p) / n )
```

That shrinks as `1/sqrt(n)`. A 40-case eval set cannot distinguish 92% from 88%
— the error bars swallow the difference. A 130-case set can. Every turn, at
**flat human cost**, the smallest regression you are able to notice gets
smaller, and the number of distinct real failures you have on file grows.

So the two things to assert are: the error bar narrows monotonically, and the
count of real production failure cases grows monotonically. Not "the number
goes up forever."

In [ ]:
@dataclass
class FlywheelState:
    turn: int = 0
    golden_size: int = 0
    labels_spent: int = 0
    history: List[Dict[str, Any]] = field(default_factory=list)

def turn_the_wheel(state: FlywheelState, golden: GoldenSet, budget: int, seed: int):
    state.turn += 1
    # each turn is a fresh week of traffic (tag keeps the prompts distinct)
    events = simulate_traffic(300, seed=seed, tag="w" + str(state.turn))
    lbls = {}
    for e in events:
        sigs = active_signals(e)
        lbls[e.req_id] = logistic(sum(WEIGHTS.get(s, 0.0) for s in sigs) + BIAS)
    ranked = sorted(events, key=lambda e: -lbls[e.req_id])
    picked = ranked[: int(budget * 0.6)]
    picked_ids = {e.req_id for e in picked}
    rng = random.Random(seed + 7)
    rest = [e for e in events if e.req_id not in picked_ids]
    rng.shuffle(rest)
    picked = picked + rest[: budget - len(picked)]     # mandatory random slice
    reviewed = [(e, "bad" if e.is_bad else "good") for e in picked]
    cases, st = build_cases(reviewed, golden)
    for c in cases:
        golden.add(c)
    state.labels_spent += budget
    state.golden_size = len(golden)
    # sensitivity: how big a lift does THIS golden set report for the v1->v2 fix?
    lift = run_eval(golden.cases, "v2", seed=seed) - run_eval(golden.cases, "v1", seed=seed)
    # statistical power: the error bar on the headline pass rate
    p = run_eval(golden.cases, "v1", seed=seed)
    se = math.sqrt(max(p * (1 - p), 1e-9) / len(golden))
    hard = sum(1 for c in golden.cases if c.topic == "multi_table_math")
    state.history.append({"turn": state.turn, "golden": len(golden),
                          "labels": state.labels_spent, "accepted": st["accepted"],
                          "detected_lift": lift, "se": se, "hard_cases": hard})
    return state

STATE = FlywheelState(golden_size=len(GOLDEN))
for k in range(3):
    turn_the_wheel(STATE, GOLDEN, budget=30, seed=100 + k)

t = Table(title="Three turns of the flywheel (30 human labels per turn)", header_style="bold")
for c in ["turn", "golden size", "cum. labels", "new cases", "real failure cases",
          "+/- 95% CI on pass rate", "detected lift (v1->v2)"]:
    t.add_column(c)
for h in STATE.history:
    t.add_row(str(h["turn"]), str(h["golden"]), str(h["labels"]), str(h["accepted"]),
              str(h["hard_cases"]), "+/-{:.1f}pp".format(196 * h["se"]),
              "[bold]{:+.1%}[/bold]".format(h["detected_lift"]))
console.print(t)

lifts = [h["detected_lift"] for h in STATE.history]
sizes = [h["golden"] for h in STATE.history]
ses = [h["se"] for h in STATE.history]
hards = [h["hard_cases"] for h in STATE.history]
print()
print("golden set grew {} -> {} cases on a FLAT budget of 30 labels/turn.".format(sizes[0], sizes[-1]))
print("real production failure cases on file: {} -> {}.".format(hards[0], hards[-1]))
print("error bar on the headline metric: +/-{:.1f}pp -> +/-{:.1f}pp  <- THIS is what compounds.".format(
    196 * ses[0], 196 * ses[-1]))
print("detected lift for the v1->v2 fix stayed ~{:+.0%} - correctly SATURATED.".format(lifts[-1]))
print("Once the set can see a bug, more copies of that bug add nothing. Power is the compounding asset.")

assert sizes == sorted(sizes) and sizes[-1] > sizes[0], "golden set must grow every turn"
assert all(h["accepted"] > 0 for h in STATE.history), "every turn must contribute new cases"
assert ses == sorted(ses, reverse=True), "the error bar must shrink monotonically - that is the payoff"
assert hards == sorted(hards) and hards[-1] > hards[0], "real failure cases on file must grow"
assert lifts[-1] > 0.10, "a matured golden set must clearly detect the fix"
assert STATE.labels_spent == 90, "human cost stayed flat at 30/turn"
print("\nOK - flat human cost, shrinking error bars, growing failure library. That is the flywheel.")

## 8. Ship it — `observability/feedback.py`

Same pattern as L71–L74: the reusable pieces go into the `observability/`
package that becomes the L76 capstone. This module adds `FeedbackEvent`,
`weak_label`, `uncertainty`, the samplers, `redact`/`content_hash`,
`GoldenSet` and `build_cases`.

In [ ]:
import os, sys, importlib

PKG = BASE + "/observability"
os.makedirs(PKG, exist_ok=True)
if not os.path.exists(PKG + "/__init__.py"):
    open(PKG + "/__init__.py", "w").write("# observability package (L71-L75)\n")

MODULE_SRC = r"""# observability/feedback.py
# Lesson 75 - Feedback loops & the data flywheel.
# Turn production signal into labeled data, and labeled data into a better eval set.

from __future__ import annotations

import hashlib
import re
from dataclasses import dataclass, field, asdict
from typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple

# ---------------------------------------------------------------- signals

# Weight of each implicit/explicit signal as evidence that a response was BAD.
# Positive weight = evidence of badness. Negative weight = evidence of goodness.
DEFAULT_WEIGHTS: Dict[str, float] = {
    "thumbs_down": 3.0,
    "thumbs_up": -3.0,
    "regenerated": 1.5,
    "edited": 1.0,
    "abandoned": 1.2,
    "guardrail_fired": 2.5,
    "low_heuristic": 1.0,
    "low_judge": 2.0,
    "high_judge": -1.5,
}


@dataclass
class FeedbackEvent:
    # One production request plus every signal we observed about it.
    req_id: str
    user_id: str
    topic: str
    text: str = ""
    heuristic_score: float = 1.0
    judge_score: Optional[float] = None
    thumbs: Optional[int] = None          # +1 / -1 / None (explicit, rare)
    regenerated: bool = False             # implicit: user hit retry
    edited: bool = False                  # implicit: user rewrote the answer
    abandoned: bool = False               # implicit: user left the session
    guardrail_fired: bool = False         # from L73 alerting
    arm: str = "champion"                 # from L74 rollout
    meta: Dict[str, Any] = field(default_factory=dict)

    def to_dict(self) -> Dict[str, Any]:
        return asdict(self)


def active_signals(ev: FeedbackEvent,
                   heuristic_floor: float = 0.55,
                   judge_floor: float = 0.5,
                   judge_ceiling: float = 0.85) -> List[str]:
    # Which named signals are present on this event.
    out: List[str] = []
    if ev.thumbs == -1:
        out.append("thumbs_down")
    if ev.thumbs == 1:
        out.append("thumbs_up")
    if ev.regenerated:
        out.append("regenerated")
    if ev.edited:
        out.append("edited")
    if ev.abandoned:
        out.append("abandoned")
    if ev.guardrail_fired:
        out.append("guardrail_fired")
    if ev.heuristic_score < heuristic_floor:
        out.append("low_heuristic")
    if ev.judge_score is not None and ev.judge_score < judge_floor:
        out.append("low_judge")
    if ev.judge_score is not None and ev.judge_score >= judge_ceiling:
        out.append("high_judge")
    return out


def _logistic(x: float) -> float:
    import math
    return 1.0 / (1.0 + math.exp(-x))


@dataclass
class WeakLabel:
    # A guess at the true label, plus how confident we are and why.
    label: str            # "bad" | "good" | "unknown"
    p_bad: float          # calibrated-ish probability the response was bad
    signals: List[str]
    def is_confident(self, lo: float = 0.25, hi: float = 0.75) -> bool:
        return self.p_bad <= lo or self.p_bad >= hi


def weak_label(ev: FeedbackEvent,
               weights: Optional[Dict[str, float]] = None,
               lo: float = 0.25,
               hi: float = 0.75) -> WeakLabel:
    # Fuse noisy signals into ONE weak label. This is a prior, never ground truth.
    w = weights or DEFAULT_WEIGHTS
    sigs = active_signals(ev)
    score = sum(w.get(s, 0.0) for s in sigs)
    p_bad = _logistic(score - 1.0)   # -1.0 bias: silence means probably fine
    if p_bad >= hi:
        lab = "bad"
    elif p_bad <= lo:
        lab = "good"
    else:
        lab = "unknown"
    return WeakLabel(label=lab, p_bad=p_bad, signals=sigs)


# ---------------------------------------------------------------- sampling

def uncertainty(ev: FeedbackEvent, wl: Optional[WeakLabel] = None) -> float:
    # Higher = more informative to send to a human. Two ingredients:
    #   1. label uncertainty  (p_bad near 0.5)
    #   2. scorer disagreement (heuristic and judge tell different stories)
    wl = wl or weak_label(ev)
    label_unc = 1.0 - 2.0 * abs(wl.p_bad - 0.5)          # 0..1, peaks at p=0.5
    if ev.judge_score is None:
        disagree = 0.0
    else:
        disagree = abs(ev.heuristic_score - ev.judge_score)
    return 0.6 * label_unc + 0.4 * disagree


def sample_for_review(events: Sequence[FeedbackEvent],
                      budget: int,
                      strategy: str = "uncertainty",
                      seed: int = 0) -> List[FeedbackEvent]:
    # Choose which `budget` events a human will actually label.
    if strategy == "random":
        import random
        rng = random.Random(seed)
        pool = list(events)
        rng.shuffle(pool)
        return pool[:budget]
    if strategy == "uncertainty":
        ranked = sorted(events, key=lambda e: -uncertainty(e))
        return list(ranked[:budget])
    if strategy == "confident_bad":
        ranked = sorted(events, key=lambda e: -weak_label(e).p_bad)
        return list(ranked[:budget])
    raise ValueError("unknown strategy: " + strategy)


def stratified_sample(events: Sequence[FeedbackEvent],
                      budget: int,
                      key=lambda e: e.topic,
                      seed: int = 0) -> List[FeedbackEvent]:
    # Even coverage across strata, so a loud topic cannot eat the whole budget.
    import random
    rng = random.Random(seed)
    buckets: Dict[Any, List[FeedbackEvent]] = {}
    for e in events:
        buckets.setdefault(key(e), []).append(e)
    for v in buckets.values():
        rng.shuffle(v)
    out: List[FeedbackEvent] = []
    keys = sorted(buckets, key=str)
    i = 0
    while len(out) < budget and any(buckets[k] for k in keys):
        k = keys[i % len(keys)]
        if buckets[k]:
            out.append(buckets[k].pop())
        i += 1
    return out


# ---------------------------------------------------------------- dataset

_PII_PATTERNS = [
    re.compile(r"[\w\.\-\+]+@[\w\-]+\.[\w\.\-]+"),
    re.compile(r"sk-[A-Za-z0-9\-_]{6,}"),
    re.compile(r"\b\d{3}-\d{2}-\d{4}\b"),
]


def redact(text: str) -> str:
    for pat in _PII_PATTERNS:
        text = pat.sub("[REDACTED]", text)
    return text


def normalize(text: str) -> str:
    return re.sub(r"\s+", " ", text.strip().lower())


def content_hash(text: str) -> str:
    return hashlib.md5(normalize(text).encode("utf-8")).hexdigest()[:16]


@dataclass
class EvalCase:
    case_id: str
    prompt: str
    topic: str
    label: str            # "bad" | "good"
    source: str           # "production" | "seed"
    provenance: Dict[str, Any] = field(default_factory=dict)


class GoldenSet:
    # The curated eval set that agent-bench runs against.
    def __init__(self, cases: Optional[Iterable[EvalCase]] = None):
        self.cases: List[EvalCase] = list(cases or [])
        self._hashes = {content_hash(c.prompt) for c in self.cases}

    def __len__(self) -> int:
        return len(self.cases)

    def contains(self, prompt: str) -> bool:
        return content_hash(prompt) in self._hashes

    def add(self, case: EvalCase) -> bool:
        # Returns False if rejected as a duplicate / already-present case.
        h = content_hash(case.prompt)
        if h in self._hashes:
            return False
        self._hashes.add(h)
        self.cases.append(case)
        return True

    def topics(self) -> Dict[str, int]:
        out: Dict[str, int] = {}
        for c in self.cases:
            out[c.topic] = out.get(c.topic, 0) + 1
        return out


def build_cases(reviewed: Sequence[Tuple[FeedbackEvent, str]],
                golden: GoldenSet,
                source: str = "production") -> Tuple[List[EvalCase], Dict[str, int]]:
    # reviewed = [(event, human_label)]. Redact, dedup within batch, and refuse
    # anything already in the golden set (test-set contamination guard).
    accepted: List[EvalCase] = []
    stats = {"seen": 0, "dup_in_batch": 0, "already_in_golden": 0, "accepted": 0}
    seen_batch = set()
    for ev, human in reviewed:
        stats["seen"] += 1
        prompt = redact(ev.text)
        h = content_hash(prompt)
        if golden.contains(prompt):
            stats["already_in_golden"] += 1
            continue
        if h in seen_batch:
            stats["dup_in_batch"] += 1
            continue
        seen_batch.add(h)
        accepted.append(EvalCase(
            case_id="case_" + h,
            prompt=prompt,
            topic=ev.topic,
            label=human,
            source=source,
            provenance={"req_id": ev.req_id, "signals": active_signals(ev), "arm": ev.arm},
        ))
        stats["accepted"] += 1
    return accepted, stats


# ---------------------------------------------------------------- flywheel

@dataclass
class FlywheelState:
    turn: int = 0
    golden_size: int = 0
    human_labels_spent: int = 0
    detected_regressions: int = 0
    history: List[Dict[str, Any]] = field(default_factory=list)

    def record(self, **kw: Any) -> None:
        row = {"turn": self.turn, "golden_size": self.golden_size,
               "human_labels_spent": self.human_labels_spent}
        row.update(kw)
        self.history.append(row)
"""

with open(PKG + "/feedback.py", "w") as f:
    f.write(MODULE_SRC)

if BASE not in sys.path:
    sys.path.insert(0, BASE)
importlib.invalidate_caches()

from observability.feedback import (FeedbackEvent as FE, weak_label as wl_fn,
                                    uncertainty as unc_fn, sample_for_review as sample_fn,
                                    stratified_sample, redact as redact_fn,
                                    content_hash as hash_fn, GoldenSet as GS,
                                    EvalCase as EC, build_cases as build_fn)

# smoke test the shipped module end to end
ev_bad = FE(req_id="r1", user_id="u1", topic="multi_table_math",
            text="join three tables mail me at a@b.com key sk-live-ZZZ999",
            heuristic_score=0.2, judge_score=0.1, thumbs=-1,
            regenerated=True, edited=True, guardrail_fired=True)
ev_good = FE(req_id="r2", user_id="u2", topic="summarize", text="summarize this",
             heuristic_score=0.95, judge_score=0.92)

assert wl_fn(ev_bad).label == "bad"
assert wl_fn(ev_good).label == "good"
assert "@" not in redact_fn(ev_bad.text) and "sk-live" not in redact_fn(ev_bad.text)
assert hash_fn("  Hello   World ") == hash_fn("hello world")

g = GS([EC(case_id="c0", prompt="summarize this", topic="summarize", label="good", source="seed")])
cases, st = build_fn([(ev_bad, "bad"), (ev_good, "good"), (ev_bad, "bad")], g)
assert st["already_in_golden"] == 1 and st["dup_in_batch"] == 1 and st["accepted"] == 1
assert unc_fn(ev_bad) < unc_fn(FE(req_id="r3", user_id="u3", topic="extract", text="x",
                                  heuristic_score=0.9, judge_score=0.2, regenerated=True))
strat = stratified_sample([ev_bad, ev_good], 2)
assert len({e.topic for e in strat}) == 2

MODULE_WRITTEN = os.path.exists(PKG + "/feedback.py")
print("wrote", PKG + "/feedback.py", "(" + str(os.path.getsize(PKG + "/feedback.py")) + " bytes)")
print("observability/ package now:", sorted(os.listdir(PKG)))
print("module smoke test: ALL PASSED")

## 9. Ten ways this goes wrong in production

| # | Pitfall | Why it bites | Fix |
|---|---|---|---|
| 1 | Treating thumbs-down as *the* label | 3% coverage, angriest-users bias | fuse many weak signals; thumbs is one input |
| 2 | Treating one implicit signal as truth | a retry can mean "wrong" *or* "give me another angle" | weights + a probability, never a hard label |
| 3 | No `unknown` bucket | forces a wrong label on genuinely ambiguous cases | three-way output; route `unknown` to humans |
| 4 | Sampling only high-`p_bad` cases for review | maximum yield, near-zero information | blend confident / uncertain / random |
| 5 | **Dropping the random slice** | you lose any unbiased estimate of production quality | random holdout is a control group, not waste |
| 6 | Mined-set pass rate reported as "production quality" | it measures your hardest traffic, not typical traffic | report both; publish the TV distance |
| 7 | No contamination guard | eval cases leak into prompt tuning / few-shot examples → score measures memorization | hash-based golden-set membership check on the write path |
| 8 | No dedup | one repeated prompt silently gets 10× weight in the headline metric | normalize + hash |
| 9 | PII in the eval set | the eval set gets committed, shared, pasted into issues | redact on the write path (L72) |
| 10 | Judge labels feeding the next generation unchecked | model collapse — metrics rise while human preference falls | re-calibrate the judge against fresh human labels every cycle |

Bonus (the subtle one): **optimizing the proxy.** "Fewer retries" is a fine
signal and a terrible objective. A model that produces confident, hard-to-check
answers reduces retries without being more correct. Any feedback signal you
optimize directly stops being a measurement — Goodhart's law with a GPU bill.

In [ ]:
checks = [
    ("explicit feedback is rare (<15% of traffic)", (n_thumbs / n) < 0.15),
    ("thumbs alone misses most failures", EXPLICIT_RECALL < 0.40),
    ("weak-label precision > 70%", WEAK_PRECISION > 0.70),
    ("signal fusion beats thumbs on recall", WEAK_RECALL > EXPLICIT_RECALL),
    ("weak labels still miss silent failures", WEAK_RECALL < 1.0),
    ("an uncertain bucket exists to review", buckets["unknown"] > 0),
    ("confident_bad beats random on yield", res["confident_bad"]["yield"] > res["random"]["yield"]),
    ("uncertainty beats confident_bad on information", res["uncertainty"]["info"] > res["confident_bad"]["info"]),
    ("blended batch draws from all 3 strategies", len(mix) == 3 and len(BLEND) == BUDGET),
    ("contamination guard fired", stats["already_in_golden"] >= 1),
    ("dedup guard fired", stats["dup_in_batch"] >= 1),
    ("no PII reached the eval set", len(pii_leaked) == 0),
    ("PAYOFF: mined set out-detects random set", MINED_LIFT > RANDOM_LIFT),
    ("mined lift is large (>25pp)", MINED_LIFT > 0.25),
    ("random set of same size barely sees it (<20pp)", RANDOM_LIFT < 0.20),
    ("mined data is measurably skewed vs random", TV_MINED > 4 * TV_RANDOM),
    ("random holdout reduces the skew", TV_COMBINED < TV_MINED),
    ("golden set grew every turn", sizes == sorted(sizes) and sizes[-1] > sizes[0]),
    ("error bar shrank every turn (power compounds)", ses == sorted(ses, reverse=True)),
    ("real failure cases on file grew", hards[-1] > hards[0]),
    ("matured golden set detects the fix", lifts[-1] > 0.10),
    ("human budget stayed flat (90 labels / 3 turns)", STATE.labels_spent == 90),
    ("observability/feedback.py written", MODULE_WRITTEN),
]

t = Table(title="Lesson 75 verification", header_style="bold")
t.add_column("#"); t.add_column("check"); t.add_column("result")
passed = 0
for i, (name, ok) in enumerate(checks, 1):
    passed += bool(ok)
    t.add_row(str(i), name, "[green]PASS[/green]" if ok else "[red]FAIL[/red]")
console.print(t)

print("\n{} / {} checks passed".format(passed, len(checks)))
assert passed == len(checks), "some checks failed"
print("Lesson 75 complete.")

## 10. Summary

| Concept | One-line version |
|---|---|
| Signal taxonomy | explicit (rare, biased) / implicit (dense, noisy) / system (hard failures only) — you need all three |
| Weak labeling | weighted signal fusion → a *probability*, with an explicit `unknown` bucket |
| Active learning | yield ≠ information; spend the human budget where the model is unsure |
| Blended sampling | confident-bad (mine failures) + uncertain (fix the labeler) + random (stay honest) |
| Dataset hygiene | redact → dedup → contamination-check, on the write path, every time |
| Distribution skew | mined data is production's tail; a random holdout is the anchor to reality |
| The flywheel | flat human cost per turn; what compounds is *statistical power* and the failure library, not the headline number |

## Homework

1. **Calibrate the weak labeler.** You now have human labels from §4. Fit the
   `WEIGHTS` dict by logistic regression against them instead of hand-picking
   numbers, then re-measure precision/recall. (This is the real version of what
   we hand-waved.)
2. **Add a session-level signal.** Using L72's `session_id`, detect *conversation
   abandonment after a specific turn* — a much stronger badness signal than a
   single-request abandon. Add it to `WEIGHTS` and see if recall improves.
3. **Close the L74 loop for real.** When a challenger arm is rolled back, auto-
   enqueue every request it served into the review queue tagged
   `source="losing_arm"`. Measure whether those cases have a higher yield than
   `confident_bad`.
4. **Build the contamination CI check.** A GitHub Action that fails the build if
   any prompt in `agent-bench`'s golden set appears in the few-shot examples or
   the prompt templates. This is the guard that actually saves you.
5. **Judge drift monitor.** Every cycle, re-score 20 previously human-labeled
   cases with the current LLM judge and alert (L73 style) if judge–human
   agreement drops. This is your model-collapse early-warning system.

## Next: Lesson 76 — Phase 8 Capstone

We consolidate `observability/` (`tracing.py`, `online_eval.py`,
`logging_search.py`, `alerting.py`, `rollout.py`, `feedback.py`) into a real
installable package, wire it into `agent-bench` end-to-end, and ship it — the
same launch-day treatment `paper-distiller` (L64) and `agent-bench` (L70) got.
That closes Phase 8 and gives you a **third** public artifact: the operations
layer that makes the first two production-grade.